# MATMED — Multi-Agent Transformer for Molecular Evolution & Design
> Research prototype for drug discovery using multi-agent RL + Transformers.

**Agents:** G-Agent (Generator) | E-Agent (Evaluator) | S-Agent (Safety) | R-Agent (Reaction) | P-Agent (Policy)


## 1. Install Dependencies

In [ ]:
pip3 install -r requirements.txt install -q rdkit-pypi torch transformers numpy pandas matplotlib
print("Dependencies installed")

## 2. Upload Project Files
Upload the following files to the Colab session:
- utils.py
- reward.py
- generator_agent.py
- evaluator_agent.py
- safety_agent.py
- reaction_agent.py
- policy_agent.py
- train_matmed.py

Or clone from your repo / Drive. Then run the cells below.

In [ ]:
# Option A: upload from local machine
# from google.colab import files
# files.upload()

# Option B: if files are in /content already, just verify
import os
required = ["utils.py","reward.py","generator_agent.py","evaluator_agent.py",
            "safety_agent.py","reaction_agent.py","policy_agent.py","train_matmed.py"]
for f in required:
    status = "OK" if os.path.exists(f) else "MISSING"
    print(f"  {f}: {status}")

## 3. Quick Agent Tests

In [ ]:
from utils import set_seed, get_zinc_sample, is_valid_smiles, SMILESTokenizer
set_seed(42)
zinc = get_zinc_sample()
print(f"ZINC sample: {len(zinc)} molecules")
print(zinc[:3])

In [ ]:
from generator_agent import GeneratorAgent, pretrain_generator
tok = SMILESTokenizer()
g = GeneratorAgent(tokenizer=tok)
print(f"G-Agent params: {sum(p.numel() for p in g.parameters()):,}")
pretrain_generator(g, zinc, num_epochs=3)
smiles_batch, embs = g.generate(batch_size=4, temperature=1.0)
print("Generated SMILES:")
for s in smiles_batch:
    print(f"  {chr(10003) if is_valid_smiles(s) else chr(10007)}  {s}")
print(f"Embedding shape: {embs.shape}")

In [ ]:
from evaluator_agent import EvaluatorAgent
e = EvaluatorAgent()
print(f"E-Agent params: {sum(p.numel() for p in e.parameters()):,}")
for smi in zinc[:3]:
    sc, emb = e.forward(smi)
    print(f"  binding={sc:.4f}  emb={emb.shape}  {smi[:35]}")

In [ ]:
from safety_agent import SafetyAgent
s = SafetyAgent(use_chemberta=False)  # set True if network available
print(f"S-Agent embed_dim={s.embed_dim}")
for smi in zinc[:3]:
    tox, admet, emb = s.forward(smi)
    print(f"  tox={tox:.4f}  admet={admet:.4f}  {smi[:35]}")

In [ ]:
from reaction_agent import ReactionAgent, compute_reaction_features
import numpy as np
r = ReactionAgent()
for smi in zinc[:3]:
    sc, emb = r.forward(smi)
    feats = compute_reaction_features(smi)
    print(f"  yield={sc:.4f}  feats={np.round(feats,3)}  {smi[:35]}")

In [ ]:
import torch
from policy_agent import PolicyAgent, ACTION_NAMES
p = PolicyAgent()
g_emb=torch.randn(256);e_emb=torch.randn(128);s_emb=torch.randn(128);r_emb=torch.randn(128)
_, probs, val = p.forward(g_emb,e_emb,s_emb,r_emb,reward=0.4)
print(f"Action probs: {probs.squeeze().detach().numpy().round(4)}")
print(f"Action names: {ACTION_NAMES}")
print(f"Value: {val.item():.4f}")

## 4. Full MATMED Training
Run the complete RL loop. Adjust  for shorter / longer runs.

In [ ]:
from reward import RewardConfig
from train_matmed import MATMEDRunner

reward_cfg = RewardConfig(alpha=1.0, beta=0.5, gamma=0.5, delta=1.0)

runner = MATMEDRunner(
    reward_config       = reward_cfg,
    num_pretrain_epochs = 5,
    lr_policy           = 3e-4,
    gamma               = 0.99,
    entropy_coeff       = 0.01,
    seed               = 42,
    use_chemberta       = False,   # flip to True if online
)

runner.train(
    num_episodes      = 30,
    steps_per_episode = 8,
    save_csv          = "matmed_metrics.csv",
)

## 5. Analyse Results

In [ ]:
import pandas as pd, matplotlib.pyplot as plt

df = pd.read_csv("matmed_metrics.csv")
print(df.tail(10).to_string(index=False))

fig, axes = plt.subplots(2, 2, figsize=(12, 8))
fig.suptitle("MATMED Training Metrics", fontsize=14, fontweight="bold")
df["avg_reward"].plot(ax=axes[0,0], title="Avg Reward", color="steelblue")
df["best_reward"].plot(ax=axes[0,1], title="Best Reward", color="darkorange")
df["pct_valid"].plot(ax=axes[1,0], title="% Valid SMILES", color="green")
df["diversity"].plot(ax=axes[1,1], title="Chemical Diversity", color="purple")
for ax in axes.flat: ax.set_xlabel("Episode"); ax.grid(alpha=0.3)
plt.tight_layout(); plt.savefig("matmed_training.png", dpi=120); plt.show()
print(f"Best molecule: {runner.best_smiles}")
print(f"Best reward  : {runner.best_reward:.4f}")

## 6. Visualise Best Molecule

In [ ]:
from rdkit import Chem; from rdkit.Chem import Draw; from IPython.display import display
best = runner.best_smiles or "CC(=O)Oc1ccccc1C(=O)O"
mol = Chem.MolFromSmiles(best)
if mol:
    display(Draw.MolToImage(mol, size=(400,300)))
    print(f"Best SMILES: {best}")
else:
    print("Invalid molecule drawn — showing aspirin instead")
    display(Draw.MolToImage(Chem.MolFromSmiles("CC(=O)Oc1ccccc1C(=O)O"), size=(400,300)))

## 7. Custom Reward Experiment

In [ ]:
from reward import RewardFunction, RewardConfig
r_fn = RewardFunction(RewardConfig(alpha=2.0, beta=0.3, gamma=0.4, delta=2.0))
b = r_fn.breakdown(binding_score=0.85, yield_score=0.60, admet_score=0.70, toxicity=0.10)
print("Reward breakdown:")
for k,v in b.items(): print(f"  {k:22s}: {v:.4f}")